Installing Phase:

In [ ]:
!pip install opencv-python
!pip install matplotlib
!pip install numpy
!pip install scikit-learn

Libraries:

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

In [ ]:
Training:

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

# ==== CONFIG ====
# ==== CONFIG ====
base_path_sess1 = '/content/drive/MyDrive/Datasets/FV_USM/Published_database_FV-USM_Dec2013/Published_database_FV-USM_Dec2013/1st_session/extractedvein'
NUM_SUBJECTS = 123
NUM_FINGERS = 4
TRAIN_IMAGES_PER_FINGER = 4
IMAGE_SIZE = (100, 300)

# ==== STORAGE ====
train_images = []
train_labels = []

# ==== STEP 1: LOAD TRAINING IMAGES ONLY ====
def load_training_strategy2():
    for subject_id in tqdm(range(1, NUM_SUBJECTS + 1), desc="Loading Training Images - S2"):
        for finger_id in range(1, NUM_FINGERS + 1):
            folder_name = f"vein{subject_id:03d}_{finger_id}"
            folder_path = os.path.join(base_path_sess1, folder_name)

            for img_idx in range(1, TRAIN_IMAGES_PER_FINGER + 1):
                img_path = os.path.join(folder_path, f"{img_idx:02d}.jpg")
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

                if img is None:
                    print(f"❌ Missing image: {img_path}")
                    continue

                img = cv2.resize(img, IMAGE_SIZE)
                img_eq = exposure.equalize_hist(img).astype(np.float64)
                img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)

                train_images.append(img_norm)
                train_labels.append(f"{subject_id:03d}_f{finger_id}_img{img_idx}")

# ==== RUN ====
load_training_strategy2()

print(f"\n✅ Loaded {len(train_images)} training images")

# ==== STEP 2: APPLY 2DPCA ON TRAINING SET ====
def compute_2dpca_projection(images_2d, num_components):
    print("\n⚙️ Computing 2DPCA projection matrix...")
    n = len(images_2d)
    h, w = images_2d[0].shape
    mean_img = sum(images_2d) / n
    G_t = np.zeros((w, w))

    for i, img in enumerate(images_2d):
        A = img - mean_img
        G_t += A.T @ A
        if i < 3:
            print(f"  ➕ Image {i+1} contribution added to covariance")

    G_t /= n
    eig_vals, eig_vecs = np.linalg.eigh(G_t)
    idx = np.argsort(-eig_vals)
    eig_vecs = eig_vecs[:, idx[:num_components]]
    print(f"✅ Computed projection matrix shape: {eig_vecs.shape}")
    return eig_vecs

# ==== PROJECT ====
num_components = 27
W = compute_2dpca_projection(train_images, num_components)

projected_train_features = []
for i, img in enumerate(train_images):
    feat = img @ W
    projected_train_features.append(feat)
    if i < 3:
        print(f"🧮 Sample {i+1} projected shape: {feat.shape}")

# ==== OPTIONAL: FLATTEN FOR CLASSIFIER ====
flat_train_features = np.array([f.flatten() for f in projected_train_features])
print(f"\n✅ Flattened feature matrix shape: {flat_train_features.shape}")


Test:

In [ ]:
# ==== STEP 3: LOAD TEST IMAGES (images 5 and 6 per finger) ====
test_images = []
test_labels = []

def load_test_strategy2():
    for subject_id in tqdm(range(1, NUM_SUBJECTS + 1), desc="Loading Test Images - S2"):
        for finger_id in range(1, NUM_FINGERS + 1):
            folder_name = f"vein{subject_id:03d}_{finger_id}"
            folder_path = os.path.join(base_path_sess1, folder_name)

            for img_idx in range(5, 7):  # test images: 5 and 6
                img_path = os.path.join(folder_path, f"{img_idx:02d}.jpg")
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

                if img is None:
                    print(f"❌ Missing test image: {img_path}")
                    continue

                img = cv2.resize(img, IMAGE_SIZE)
                img_eq = exposure.equalize_hist(img).astype(np.float64)
                img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)

                test_images.append(img_norm)
                test_labels.append(f"{subject_id:03d}_f{finger_id}_img{img_idx}")

# ==== RUN TEST LOADING ====
load_test_strategy2()

# ==== PROJECT TEST IMAGES USING TRAINED W ====
projected_test_features = []
for i, img in enumerate(test_images):
    feat = img @ W
    projected_test_features.append(feat)

flat_test_features = np.array([f.flatten() for f in projected_test_features])

# ==== STEP 4: CLASSIFICATION USING MANHATTAN DISTANCE ====
from scipy.spatial.distance import cdist

print("\n🔍 Classifying test samples using Manhattan (L1) distance...")

dist_matrix = cdist(flat_test_features, flat_train_features, metric='cityblock')
predicted_indices = np.argmin(dist_matrix, axis=1)
predicted_labels = np.array(train_labels)[predicted_indices]

# ==== STEP 5: EVALUATION ====
correct = 0
for true_label, pred_label in zip(test_labels, predicted_labels):
    true_subject = "_".join(true_label.split("_")[:2])  # e.g., '001_f1'
    pred_subject = "_".join(pred_label.split("_")[:2])
    if true_subject == pred_subject:
        correct += 1

accuracy = 100 * correct / len(test_labels)
print(f"\n✅ Recognition accuracy = {accuracy:.2f}% ({correct} / {len(test_labels)} correct)")
